<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Avalon-MM Bus Helper

`AvalonMMMasterBFM` is a lightweight cocotb host for single-beat Avalon-MM register access.

In [ ]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

from fpga_verification.sim.buses import AvalonMMBus, AvalonMMMasterBFM


@cocotb.test()
async def control_register_test(dut):
    cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())

    avmm = AvalonMMMasterBFM.from_prefix(
        dut,
        "control",
        dut.clk,
        reset=dut.reset,
        default_byteenable=0xF,
    )
    avmm.init_idle()

    dut.reset.value = 1
    await RisingEdge(dut.clk)
    dut.reset.value = 0
    await avmm.wait_reset_release(active_value=1)

    await avmm.write(0x00, 0x00000001, timeout_cycles=32)
    status = await avmm.read(0x04, timeout_cycles=32)
    await avmm.wait_set(0x04, 0x1, timeout_cycles=256)

    assert status & 0x1 in (0, 1)

In [ ]:
# Manual bus construction is useful when DUT signal names do not share a prefix.
bus = AvalonMMBus(
    address=dut.ctrl_address,
    writedata=dut.ctrl_writedata,
    write=dut.ctrl_write,
    read=dut.ctrl_read,
    readdata=dut.ctrl_readdata,
    waitrequest=getattr(dut, "ctrl_waitrequest", None),
    readdatavalid=getattr(dut, "ctrl_readdatavalid", None),
    byteenable=getattr(dut, "ctrl_byteenable", None),
)
avmm = AvalonMMMasterBFM(bus, dut.clk, reset=dut.reset)

Supported optional Avalon-MM signals: `waitrequest`, `readdatavalid`, and `byteenable`. The helper does not issue bursts or multiple outstanding reads.